# features.ipynb
Builds the feature tensor and cross-sectionally ranked target for the GNN.

## Features (15 total per stock)
| # | Name | Description |
|---|------|-------------|
| 1 | roll_mean_5 | 5-day rolling mean log return |
| 2 | roll_mean_20 | 20-day rolling mean log return |
| 3 | roll_std_5 | 5-day rolling volatility |
| 4 | roll_std_20 | 20-day rolling volatility |
| 5 | mom_12_1 | 12-month momentum, skip last month |
| 6 | mom_6_1 | 6-month momentum, skip last month |
| 7 | mom_3_1 | 3-month momentum, skip last month |
| 8 | mom_1_0 | 1-month momentum |
| 9 | reversal_1w | 1-week short-term reversal (sign-flipped) |
| 10 | sharpe_20 | 20-day Sharpe ratio |
| 11 | sharpe_60 | 60-day Sharpe ratio |
| 12 | rel_strength_20 | Return vs equal-weight market, 20-day |
| 13 | rel_strength_60 | Return vs equal-weight market, 60-day |
| 14 | vol_weighted_mom_5 | Volume-weighted momentum, 5-day |
| 15 | vol_weighted_mom_20 | Volume-weighted momentum, 20-day |

## Diagnostic order
1. Build raw features
2. Run IC + logistic diagnostics on raw features
3. Apply cross-sectional z-score normalization
4. Save normalized features

In [63]:
import pandas as pd
import numpy as np
from scipy.stats import rankdata, spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import os

In [64]:
# ── Load data ─────────────────────────────────────────────────────────────────
log_returns = pd.read_csv("data/log_returns.csv", index_col=0, parse_dates=True)

VOLUME_PATH = "data/volume.csv"
has_volume  = os.path.exists(VOLUME_PATH)
if has_volume:
    volume = pd.read_csv(VOLUME_PATH, index_col=0, parse_dates=True)
    volume = volume.reindex(columns=log_returns.columns)
    print("volume loaded")
else:
    print("WARNING: data/volume.csv not found — volume features will be plain momentum.")
    volume = None

print(f"log_returns shape: {log_returns.shape}")

volume loaded
log_returns shape: (2764, 462)


In [65]:
# ── Build all features ────────────────────────────────────────────────────────
r   = log_returns
EPS = 1e-8

# Rolling mean & std
roll_mean_5  = r.rolling(5).mean()
roll_mean_20 = r.rolling(20).mean()
roll_std_5   = r.rolling(5).std()
roll_std_20  = r.rolling(20).std()

# Momentum at multiple horizons (all skip last month to avoid reversal)
mom_12_1    = r.shift(20).rolling(230).sum()
mom_6_1     = r.shift(20).rolling(105).sum()
mom_3_1     = r.shift(20).rolling(45).sum()
mom_1_0     = r.shift(1).rolling(20).sum()
reversal_1w = -r.shift(1).rolling(5).sum()

# Sharpe ratio signal
sharpe_20 = r.rolling(20).mean() / (r.rolling(20).std() + EPS)
sharpe_60 = r.rolling(60).mean() / (r.rolling(60).std() + EPS)

# Relative strength vs equal-weight market
market_ret_20   = r.rolling(20).sum().mean(axis=1)
market_ret_60   = r.rolling(60).sum().mean(axis=1)
rel_strength_20 = r.rolling(20).sum().subtract(market_ret_20, axis=0).reindex(r.index)
rel_strength_60 = r.rolling(60).sum().subtract(market_ret_60, axis=0).reindex(r.index)

# Volume-weighted momentum
if has_volume:
    rel_vol             = volume / (volume.rolling(20).mean() + EPS)
    vol_weighted_mom_5  = ((r * rel_vol).rolling(5).sum()  / rel_vol.rolling(5).sum()).reindex(r.index)
    vol_weighted_mom_20 = ((r * rel_vol).rolling(20).sum() / rel_vol.rolling(20).sum()).reindex(r.index)
else:
    vol_weighted_mom_5  = r.shift(1).rolling(5).sum()
    vol_weighted_mom_20 = r.shift(1).rolling(20).sum()

print("all features computed")

all features computed


In [66]:
# ── Assemble into raw feature tensor (T, N, F) ────────────────────────────────
feature_list = [
    roll_mean_5, roll_mean_20,
    roll_std_5,  roll_std_20,
    mom_12_1, mom_6_1, mom_3_1, mom_1_0,
    reversal_1w,
    sharpe_20, sharpe_60,
    rel_strength_20, rel_strength_60,
    vol_weighted_mom_5, vol_weighted_mom_20,
]

feature_names = [
    "roll_mean_5", "roll_mean_20",
    "roll_std_5",  "roll_std_20",
    "mom_12_1", "mom_6_1", "mom_3_1", "mom_1_0",
    "reversal_1w",
    "sharpe_20", "sharpe_60",
    "rel_strength_20", "rel_strength_60",
    "vol_weighted_mom_5", "vol_weighted_mom_20",
]

# Confirm shapes match
for name, f in zip(feature_names, feature_list):
    print(name, f.shape)

stacked    = np.stack([f.values for f in feature_list], axis=-1)  # (T, N, F)
valid_mask = ~np.any(np.isnan(stacked), axis=(1, 2))               # rows with no NaN

dates_all     = feature_list[0].index
dates_valid   = dates_all[valid_mask]
stacked_final = stacked[valid_mask]                                # raw, not normalized yet

print(f"\nRaw feature tensor shape: {stacked_final.shape}  (T, N, F)")

roll_mean_5 (2764, 462)
roll_mean_20 (2764, 462)
roll_std_5 (2764, 462)
roll_std_20 (2764, 462)
mom_12_1 (2764, 462)
mom_6_1 (2764, 462)
mom_3_1 (2764, 462)
mom_1_0 (2764, 462)
reversal_1w (2764, 462)
sharpe_20 (2764, 462)
sharpe_60 (2764, 462)
rel_strength_20 (2764, 462)
rel_strength_60 (2764, 462)
vol_weighted_mom_5 (2764, 462)
vol_weighted_mom_20 (2764, 462)

Raw feature tensor shape: (2023, 462, 15)  (T, N, F)


In [67]:
# Target: next-month realized volatility (std of daily returns over next 20 days)
target_raw     = log_returns.shift(-20).rolling(20).std()
target_aligned = target_raw.reindex(dates_valid)
target_valid   = target_aligned.dropna(how="all")
T_target       = len(target_valid)

stacked_final = stacked_final[:T_target]
dates_final   = dates_valid[:T_target]

print(f"Target shape:          {target_valid.shape}")
print(f"Feature tensor shape:  {stacked_final.shape}")

valid_counts = (~np.isnan(target_valid.values)).sum(axis=1)
print(f"Min valid stocks per timestep: {valid_counts.min()}")
print(f"Max valid stocks per timestep: {valid_counts.max()}")

Target shape:          (2003, 462)
Feature tensor shape:  (2003, 462, 15)
Min valid stocks per timestep: 462
Max valid stocks per timestep: 462


In [68]:
# Cross-sectional rank target scaled to [-1, 1]
# For volatility: rank 1 = lowest vol stock, rank -1 = highest vol stock
# (we flip so high predicted rank = low vol = desirable)
n_timesteps, n_stocks = target_valid.values.shape
targets_ranked = np.zeros_like(target_valid.values, dtype=np.float32)

for t in range(n_timesteps):
    row       = target_valid.values[t]
    valid_idx = ~np.isnan(row)
    if valid_idx.sum() < 2:
        continue
    ranks             = np.empty_like(row)
    ranks[valid_idx]  = rankdata(row[valid_idx]) - 1
    ranks[~valid_idx] = np.nan
    n = valid_idx.sum()
    targets_ranked[t, valid_idx] = (ranks[valid_idx] / (n - 1)) * 2 - 1

print(f"Ranked target shape: {targets_ranked.shape}")
print(f"  Min:  {np.nanmin(targets_ranked):.4f}")
print(f"  Max:  {np.nanmax(targets_ranked):.4f}")
print(f"  Mean: {np.nanmean(targets_ranked):.4f}")
print(f"  Std:  {np.nanstd(targets_ranked):.4f}")

Ranked target shape: (2003, 462)
  Min:  -1.0000
  Max:  1.0000
  Mean: 0.0000
  Std:  0.5786


In [69]:
# ── IC diagnostic (on RAW features) ──────────────────────────────────────────
# Mean IC > 0.02 = usable signal. ICIR > 0.3 = consistent signal.

T             = stacked_final.shape[0]
ic_by_feature = np.zeros((T, len(feature_names)))

for t in range(T):
    target_row = target_valid.values[t]
    valid      = ~np.isnan(target_row)
    if valid.sum() < 50:
        continue
    for f_idx in range(len(feature_names)):
        feat_row   = stacked_final[t, :, f_idx]
        feat_valid = valid & ~np.isnan(feat_row)
        if feat_valid.sum() < 50:
            continue
        ic, _ = spearmanr(feat_row[feat_valid], target_row[feat_valid])
        ic_by_feature[t, f_idx] = ic

mean_ic = np.nanmean(ic_by_feature, axis=0)
ic_std  = np.nanstd(ic_by_feature,  axis=0)
icir    = mean_ic / (ic_std + 1e-8)

print(f"{'Feature':<25}  {'Mean IC':>8}  {'IC Std':>8}  {'ICIR':>8}")
print("-" * 55)
for name, mic, std, ir in zip(feature_names, mean_ic, ic_std, icir):
    print(f"{name:<25}  {mic:>8.4f}  {std:>8.4f}  {ir:>8.4f}")

Feature                     Mean IC    IC Std      ICIR
-------------------------------------------------------
roll_mean_5                 -0.0395    0.2394   -0.1652
roll_mean_20                -0.0665    0.2274   -0.2926
roll_std_5                   0.3998    0.1256    3.1831
roll_std_20                  0.5390    0.1071    5.0350
mom_12_1                    -0.0590    0.2607   -0.2264
mom_6_1                     -0.0533    0.2494   -0.2135
mom_3_1                     -0.0506    0.2353   -0.2151
mom_1_0                     -0.0640    0.2275   -0.2815
reversal_1w                  0.0386    0.2389    0.1615
sharpe_20                   -0.1117    0.2038   -0.5482
sharpe_60                   -0.1494    0.2042   -0.7317
rel_strength_20             -0.0665    0.2274   -0.2926
rel_strength_60             -0.0855    0.2398   -0.3566
vol_weighted_mom_5          -0.0354    0.2255   -0.1571
vol_weighted_mom_20         -0.0561    0.1992   -0.2818


In [73]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score

X = stacked_final.reshape(-1, stacked_final.shape[-1])
y = target_valid.values.reshape(-1)

mask = ~(np.isnan(X).any(axis=1) | np.isnan(y))
X, y = X[mask], y[mask]

split = int(0.7 * len(X))
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
pred_val = ridge.predict(X_val)

baseline_mse = mean_squared_error(y_val, np.full_like(y_val, y_train.mean()))
val_mse      = mean_squared_error(y_val, pred_val)
val_r2       = r2_score(y_val, pred_val)

print(f"Baseline MSE (predict train mean): {baseline_mse:.6f}")
print(f"Ridge Val MSE:                     {val_mse:.6f}")
print(f"Ridge Val R2:                      {val_r2:.6f}")
print(f"Improvement over baseline:         {(baseline_mse - val_mse) / baseline_mse * 100:.2f}%")
print()
print("Coefficients:")
for name, coef in zip(feature_names, ridge.coef_):
    print(f"  {name:<25}: {coef:.6f}")

Baseline MSE (predict train mean): 0.000090
Ridge Val MSE:                     0.000075
Ridge Val R2:                      0.141539
Improvement over baseline:         16.42%

Coefficients:
  roll_mean_5              : -0.221579
  roll_mean_20             : -0.290865
  roll_std_5               : 0.162569
  roll_std_20              : 0.360398
  mom_12_1                 : -0.002544
  mom_6_1                  : 0.002754
  mom_3_1                  : 0.000251
  mom_1_0                  : 0.000841
  reversal_1w              : 0.005047
  sharpe_20                : -0.001126
  sharpe_60                : -0.002410
  rel_strength_20          : 0.022226
  rel_strength_60          : -0.002651
  vol_weighted_mom_5       : 0.076809
  vol_weighted_mom_20      : 0.012803


In [71]:
# ── Cross-sectional z-score normalization (AFTER diagnostics) ─────────────────
stacked_norm = stacked_final.copy()

for t in range(stacked_norm.shape[0]):
    for f in range(stacked_norm.shape[2]):
        col    = stacked_norm[t, :, f]
        mu     = np.nanmean(col)
        sigma  = np.nanstd(col)
        stacked_norm[t, :, f] = (col - mu) / (sigma + 1e-8)

print(f"Normalized feature tensor shape: {stacked_norm.shape}")

Normalized feature tensor shape: (2003, 462, 15)


In [72]:
# ── Save artefacts ────────────────────────────────────────────────────────────
os.makedirs("data/processed", exist_ok=True)

np.save("data/processed/features.npy",       stacked_norm.astype(np.float32))
np.save("data/processed/targets_ranked.npy", targets_ranked)
np.save("data/processed/targets_raw.npy",    target_valid.values.astype(np.float32))

pd.Series(dates_final).to_csv("data/processed/dates.csv",           index=False)
pd.Series(log_returns.columns.tolist()).to_csv("data/processed/tickers.csv",       index=False)
pd.Series(feature_names).to_csv("data/processed/feature_names.csv", index=False)

print("Saved:")
print(f"  features.npy       {stacked_norm.shape}  (normalized)")
print(f"  targets_ranked.npy {targets_ranked.shape}")
print(f"  targets_raw.npy    {target_valid.values.shape}")
print(f"  dates.csv, tickers.csv, feature_names.csv")

Saved:
  features.npy       (2003, 462, 15)  (normalized)
  targets_ranked.npy (2003, 462)
  targets_raw.npy    (2003, 462)
  dates.csv, tickers.csv, feature_names.csv
